Scratch/testing document for `getMeanStd.py` and `module_riskModel.py`

In [8]:
#!/usr/bin/env python
import numpy as np
import calendar
import random
import time
import sys
import dask.array as da
import os
import gc
import xarray as xr
import copy
import pandas as pd
import Namelist as gv
import netCDF4 as nc
from netCDF4 import Dataset
from scipy import stats
from tools.util import int2str,date_interpolation
from scipy.io import loadmat, netcdf_file
from datetime import datetime,timedelta
from tools.regression4 import calMultiRegression_coef_mean_water
from tools.regression4 import calMultiRegression_coef_mean_land

### TEMPORARY (hopefully) code snippet so that the HTML view of datasets/dataarrays is viewable in darkmode on jupyter lab/VS code
from IPython.display import HTML, display

display(HTML("""
<style>
    .xr-wrap {
        --xr-background-color-row-odd: rgba(100, 100, 100, 0.10) !important;
        --xr-background-color-row-even: rgba(30, 30, 30, 0.10) !important;
    }
</style>
"""))

In [9]:
"""
Read in global data and make the new bt with same length
this step can be elimated if we are using ibtracks in the future CHAZ development	
"""
basinName = ['atl','wnp','enp','ni','sh']
nd = 0

## not sure if this is the correct indentation:
for iib in range(0,len(basinName),1):
    ib = basinName[iib]
    f = gv.ipath + 'bt_'+ib+'.nc'          

    #bt1 = nc.Dataset(f)
    bt1 = xr.open_dataset(f)
    if iib == 0:
        maxCol = bt1['PIslp'].shape[0]
    else:
        maxCol = np.nanmax([maxCol,bt1['PIslp'].shape[0]])
    ## in bt1, the time is datenumber start from 1800,01,01,0,0. 
    ## So if datenumber is 0 means there is no data
    nd += bt1['PIslp'].shape[1]

bt = {}
for iib in range(0,len(basinName),1):
    ib = basinName[iib]
    f =gv.ipath + 'bt_'+ib+'.nc'   
    print(f"Opening: {f}")          # confirm this actually runs
    bt1 = xr.open_dataset(f)
    print(f"Opened OK: {ib}")       # confirm assignment succeeded
    for iv in bt1.variables.keys():
        print(iv)
        if iib == 0:
            if np.size(bt1.variables[iv].shape) >1:
                 bt[iv] = np.zeros([maxCol,bt1.variables[iv].shape[1]])*float('nan')

Opening: input/bt_atl.nc
Opened OK: atl
PIslp
PIslpMean
PIwspd
PIwspdMean
StormId
StormInitMonth
StormLat
StormLon
StormMslp
StormMwspd
StormYear
T100
T100Mean
T200
T200Mean
Time
UShear
UShearMean
VShear
VShearMean
dPIwspd
dPdt
dThetaE
dThetaEMean
dThetaEs
dThetaEsMean
dVdt
div200
div200Mean
landmask
landmaskMean
rh
rh500_300
rhMean
trDir
trSpeed
Opening: input/bt_wnp.nc
Opened OK: wnp
PIslp
PIslpMean
PIwspd
PIwspdMean
StormId
StormInitMonth
StormLat
StormLon
StormMslp
StormMwspd
StormYear
T100
T100Mean
T200
T200Mean
Time
UShear
UShearMean
VShear
VShearMean
dPIwspd
dPdt
dThetaE
dThetaEMean
dThetaEs
dThetaEsMean
dVdt
div200
div200Mean
landmask
landmaskMean
rh
rh500_300
rhMean
trDir
trSpeed
Opening: input/bt_enp.nc
Opened OK: enp
PIslp
PIslpMean
PIwspd
PIwspdMean
StormId
StormInitMonth
StormLat
StormLon
StormMslp
StormMwspd
StormYear
T100
T100Mean
T200
T200Mean
Time
UShear
UShearMean
VShear
VShearMean
dPIwspd
dPdt
dThetaE
dThetaEMean
dThetaEs
dThetaEsMean
dVdt
div200
div200Mean
landmask


In [11]:
bt1.PIwspdMean

<xarray.DataArray 'PIwspdMean' (d1: 130, d2: 1918)> Size: 2MB
[249340 values with dtype=float64]
Dimensions without coordinates: d1, d2